# V18 Supplementary Table S1/S2 Verification
**Purpose:** Cross-check S1 (29 pathway definitions) and S2 (196 gene candidates) against actual Colab output CSV files.

Run cells sequentially. Results will be saved to .

In [1]:
# =================================================================
# CELL 1: SETUP & MOUNT
# =================================================================
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import numpy as np
import json, os
from collections import Counter

BASE = '/content/drive/MyDrive/ITLAS/results/version18-analysis'
C3_DIR = f'{BASE}/C3_gene_expression'
C4_DIR = f'{BASE}/C4_pathway'
C5_DIR = f'{BASE}/C5_genes'
C9_DIR = f'{BASE}/C9_method_fixes'

print("=" * 70)
print("  S1/S2 VERIFICATION — Reading actual Colab outputs")
print("=" * 70)

# List available files
for folder, label in [(C3_DIR, 'C3'), (C4_DIR, 'C4'), (C5_DIR, 'C5'), (C9_DIR, 'C9')]:
    if os.path.exists(folder):
        files = os.listdir(folder)
        print(f"\n  {label}: {len(files)} files")
        for f in sorted(files)[:10]:
            print(f"    {f}")
    else:
        print(f"\n  ⚠️ {label}: folder not found at {folder}")

print("\n✅ Cell 1 complete")

Mounted at /content/drive
  S1/S2 VERIFICATION — Reading actual Colab outputs

  C3: 9 files
    C3_NL_vs_IT_all.csv
    C3_NL_vs_IT_significant.csv
    C3_all_statistics.csv.gz
    C3_blood_donor_gene_expression.csv.gz
    C3_gene_list_196genes.csv
    C3_liver_blood_discrepancy_NL_IT.csv
    C3_liver_donor_gene_expression.csv.gz
    C3_pattern_classification.csv
    figures

  C4: 4 files
    C4_pathway_blood.csv
    C4_pathway_liver.csv
    C4_selected_genes_for_C5.csv
    C4_selected_pathways_for_C5.csv

  C5: 4 files
    C5_all_significant_genes.csv
    C5_gene_significance_ranking.csv
    C5_genes_blood.csv
    C5_genes_liver.csv

  C9: 7 files
    Fix1B_StratifiedFDR
    Fix1_FDR
    Fix2_NewPathways
    Fix3B_C3onlyGenes
    Fix3_ExtremeFC
    Fix4_GeneReconcile
    Fix5_BootstrapCI

✅ Cell 1 complete


In [ ]:
# =================================================================
# CELL 2: LOAD C3 GENE LIST (196 genes — GROUND TRUTH)
# =================================================================
print("=" * 70)
print("  LOADING C3 GENE LIST (196 genes)")
print("=" * 70)

c3_gene_file = f'{C3_DIR}/C3_gene_list_196genes.csv'
if os.path.exists(c3_gene_file):
    df_c3_genes = pd.read_csv(c3_gene_file)
    print(f"  C3 gene list: {len(df_c3_genes)} rows")
    print(f"  Columns: {list(df_c3_genes.columns)}")
    print(f"\n  First 10 genes:")
    print(df_c3_genes.head(10).to_string(index=False))
    C3_GENES = sorted(df_c3_genes.iloc[:, 0].tolist())  # first column = gene names
    print(f"\n  Total C3 genes: {len(C3_GENES)}")
else:
    print(f"  ⚠️ File not found: {c3_gene_file}")
    print("  Trying alternative: extract from C3_all_statistics.csv.gz")
    c3_stats = f'{C3_DIR}/C3_all_statistics.csv.gz'
    if os.path.exists(c3_stats):
        df_c3 = pd.read_csv(c3_stats)
        C3_GENES = sorted(df_c3['gene'].unique().tolist())
        print(f"  Extracted {len(C3_GENES)} unique genes from C3_all_statistics")
    else:
        C3_GENES = []
        print("  🔴 Cannot find C3 gene list!")

print(f"\n✅ C3 genes loaded: {len(C3_GENES)}")

  LOADING C3 GENE LIST (196 genes)
  C3 gene list: 196 rows
  Columns: ['gene', 'in_C3', 'in_C3b', 'source']

  First 10 genes:
   gene  in_C3  in_C3b source
 ACADVL   True   False     C3
  AICDA  False    True    C3b
   AIM2   True    True C3+C3b
   AKT1   True   False     C3
    APC   True   False     C3
    ATM   True   False     C3
ATP5F1A   True   False     C3
    ATR   True   False     C3
  AXIN1   True   False     C3
    B2M   True   False     C3

  Total C3 genes: 196

✅ C3 genes loaded: 196


In [ ]:
# =================================================================
# CELL 3: LOAD C5 GENE LIST (148 genes — GROUND TRUTH)
# =================================================================
print("=" * 70)
print("  LOADING C5 GENE LIST (148 genes)")
print("=" * 70)

# Try C4_selected_genes_for_C5.csv first (this defines which genes went into C5)
c4_sel_file = f'{C4_DIR}/C4_selected_genes_for_C5.csv'
if os.path.exists(c4_sel_file):
    df_c4_sel = pd.read_csv(c4_sel_file)
    print(f"  C4 selected genes: {len(df_c4_sel)} rows")
    print(f"  Columns: {list(df_c4_sel.columns)}")
    C5_GENES_from_C4 = sorted(df_c4_sel.iloc[:, 0].tolist())
    print(f"  Total from C4 selection: {len(C5_GENES_from_C4)}")
else:
    C5_GENES_from_C4 = []
    print(f"  ⚠️ {c4_sel_file} not found")

# Also extract from C5 results directly
c5_liver = f'{C5_DIR}/C5_genes_liver.csv'
c5_blood = f'{C5_DIR}/C5_genes_blood.csv'
C5_GENES_from_results = set()
for f in [c5_liver, c5_blood]:
    if os.path.exists(f):
        df = pd.read_csv(f)
        print(f"\n  {os.path.basename(f)}: {len(df)} rows, columns: {list(df.columns)[:8]}")
        # Find gene column
        gene_col = None
        for candidate in ['gene', 'Gene', 'gene_name', 'symbol']:
            if candidate in df.columns:
                gene_col = candidate
                break
        if gene_col is None and len(df.columns) > 0:
            # Try first column
            gene_col = df.columns[0]
        if gene_col:
            genes = df[gene_col].unique().tolist()
            C5_GENES_from_results.update(genes)
            print(f"    Unique genes: {len(genes)}")

C5_GENES_from_results = sorted(C5_GENES_from_results)
print(f"\n  C5 genes from C4 selection: {len(C5_GENES_from_C4)}")
print(f"  C5 genes from C5 results:   {len(C5_GENES_from_results)}")

# Use whichever is available; prefer C4 selection as it's the definitive list
C5_GENES = C5_GENES_from_C4 if C5_GENES_from_C4 else C5_GENES_from_results
print(f"\n✅ C5 genes loaded: {len(C5_GENES)}")

  LOADING C5 GENE LIST (148 genes)
  C4 selected genes: 148 rows
  Columns: ['gene']
  Total from C4 selection: 148

  C5_genes_liver.csv: 7252 rows, columns: ['tissue', 'gene', 'lineage', 'pathway', 'comparison', 'stage1', 'stage2', 'mean_s1']
    Unique genes: 148

  C5_genes_blood.csv: 7252 rows, columns: ['tissue', 'gene', 'lineage', 'pathway', 'comparison', 'stage1', 'stage2', 'mean_s1']
    Unique genes: 148

  C5 genes from C4 selection: 148
  C5 genes from C5 results:   148

✅ C5 genes loaded: 148


In [ ]:
# =================================================================
# CELL 4: LOAD C2/C4 PATHWAY DEFINITIONS (26 original)
# =================================================================
print("=" * 70)
print("  LOADING PATHWAY DEFINITIONS")
print("=" * 70)

# The 26 original pathways are defined in C2 code.
# We reconstruct them here and verify against C4 pathway results.
GENE_SETS_26 = {
    'inflammasome':     ['NLRP3', 'CASP1', 'IL1B', 'IL18', 'PYCARD', 'GSDMD'],
    'cytotoxicity':     ['GZMB', 'GZMA', 'GZMK', 'PRF1', 'GNLY', 'NKG7', 'FASLG'],
    'checkpoint':       ['PDCD1', 'CTLA4', 'HAVCR2', 'LAG3', 'TIGIT', 'TOX'],
    'exhaustion':       ['TOX', 'PDCD1', 'HAVCR2', 'LAG3', 'TIGIT', 'ENTPD1'],
    'nk_function':      ['NCAM1', 'KLRD1', 'KLRC2', 'KLRK1', 'NCR1', 'NCR3'],
    'nk_il15_dual':     ['IL2RB', 'IL2RG', 'GZMB', 'PRF1', 'GNLY', 'FCGR3A', 'KLRD1', 'KLRC2'],
    'il15_mtor':        ['IL2RB', 'IL2RG', 'JAK1', 'JAK3', 'STAT5A', 'STAT5B', 'MTOR', 'RPTOR', 'RPS6KB1', 'EIF4EBP1'],
    'immune_evasion':   ['CD274', 'PDCD1LG2', 'LGALS9', 'IDO1', 'TGFB1', 'IL10', 'HAVCR2', 'VTCN1'],
    'treg':             ['FOXP3', 'IL2RA', 'CTLA4', 'IKZF2', 'TNFRSF18'],
    'naive_t':          ['LEF1', 'TCF7', 'CCR7', 'SELL', 'IL7R'],
    'memory_t':         ['IL7R', 'CD44', 'EOMES', 'BCL6', 'ID3', 'PRDM1', 'TCF7', 'GZMK'],
    'tf_programs':      ['TBX21', 'EOMES', 'GATA3', 'RORC', 'BCL6', 'PRDM1', 'IRF4', 'BATF', 'MAF'],
    'tissue_resident':  ['ITGAE', 'CXCR6', 'ZNF683', 'PRDM1', 'RUNX3'],
    'stemness':         ['TCF7', 'LEF1', 'MYB', 'KLF2', 'SELL', 'IL7R', 'BCL2'],
    'glycolysis':       ['HK1', 'HK2', 'PFKM', 'PKM', 'LDHA', 'SLC2A1', 'ENO1', 'GAPDH'],
    'oxphos':           ['ATP5F1A', 'ATP5F1B', 'NDUFA1', 'NDUFB1', 'UQCRC1', 'COX4I1', 'SDHB'],
    'mito_dysfunction': ['MT-ND1', 'MT-ND2', 'MT-CO1', 'MT-CO2', 'MT-ATP6', 'MT-CYB', 'PINK1', 'PRKN'],
    'metabolic_recovery': ['PPARGC1A', 'TFAM', 'NRF1', 'ESRRA', 'SIRT1', 'SIRT3'],
    'cancer_associated': ['ATM', 'BCL2', 'ZEB1', 'SNAI1', 'TWIST1', 'CDH1', 'VIM', 'MYC', 'CCND1'],
    'fibrosis':         ['COL1A1', 'COL3A1', 'ACTA2', 'FAP', 'TGFB1', 'TGFB2', 'TGFBR1', 'TGFBR2', 'CTGF', 'LOX'],
    'senescence':       ['CDKN1A', 'CDKN2A', 'GLB1', 'TP53', 'RB1', 'SERPINE1', 'IGFBP7'],
    'epigenetics':      ['TET2', 'TOX', 'EZH2', 'DNMT1', 'DNMT3A', 'DNMT3B', 'HDAC1', 'KDM6A'],
    'angiogenesis':     ['HIF1A', 'VEGFA', 'VEGFB', 'KDR', 'FLT1', 'ANGPT1', 'ANGPT2', 'TEK', 'PECAM1'],
    'cell_cycle':       ['MKI67', 'TOP2A', 'PCNA', 'CDK1', 'CDK2', 'CDK4', 'CCNB1', 'CCND1', 'CCNE1', 'RB1'],
    'proliferation':    ['MKI67', 'TOP2A', 'PCNA', 'CDK1', 'CCNB1'],
    'apoptosis':        ['BCL2', 'BAX', 'BAK1', 'CASP3', 'CASP8', 'FAS'],
}

# 3 new pathways from C9
GENE_SETS_3NEW = {
    'antigen_presentation': ['HLA-DRA', 'HLA-DRB1', 'HLA-DPB1', 'HLA-DPA1', 'HLA-DQB1', 'CD74', 'B2M', 'TAP1', 'TAP2', 'CIITA'],
    'type1_ifn':            ['MX1', 'ISG15', 'STAT1', 'STAT2', 'IRF3', 'IRF7', 'IFNAR1', 'OAS1', 'IFIT1', 'DDX58'],
    'tgfb_signaling':       ['TGFB1', 'TGFB2', 'TGFBR1', 'TGFBR2', 'SMAD2', 'SMAD3', 'SMAD4', 'SMAD7', 'ACVR1', 'LTBP1'],
}

ALL_29 = {**GENE_SETS_26, **GENE_SETS_3NEW}

# Verify against C4 pathway results
c4_pw_file = f'{C4_DIR}/C4_pathway_liver.csv'
if os.path.exists(c4_pw_file):
    df_c4 = pd.read_csv(c4_pw_file)
    c4_pathways = sorted(df_c4['pathway'].unique().tolist()) if 'pathway' in df_c4.columns else []
    print(f"  C4 liver pathways in CSV: {len(c4_pathways)}")
    print(f"  Our 26 defined: {len(GENE_SETS_26)}")

    # Check if C4 CSV pathways match our definitions
    our_26 = set(GENE_SETS_26.keys())
    csv_set = set(c4_pathways)
    if our_26 == csv_set:
        print("  ✅ C4 pathway names MATCH our 26 definitions exactly")
    else:
        missing = our_26 - csv_set
        extra = csv_set - our_26
        if missing: print(f"  ⚠️ In our list but NOT in C4 CSV: {missing}")
        if extra: print(f"  ⚠️ In C4 CSV but NOT in our list: {extra}")

# Unique gene counts
all_pw_genes_26 = set()
for genes in GENE_SETS_26.values():
    all_pw_genes_26.update(genes)

all_pw_genes_29 = set()
for genes in ALL_29.values():
    all_pw_genes_29.update(genes)

# Multi-pathway genes
gene_count = Counter()
for pw, genes in ALL_29.items():
    for g in genes:
        gene_count[g] += 1
multi = {g for g, c in gene_count.items() if c > 1}

print(f"\n  26 original pathways → {len(all_pw_genes_26)} unique genes")
print(f"  29 pathways (with 3 new) → {len(all_pw_genes_29)} unique genes")
print(f"  Genes in ≥2 pathways: {len(multi)}")
print(f"  Multi-pathway genes: {sorted(multi)}")

print(f"\n✅ Cell 4 complete")

  LOADING PATHWAY DEFINITIONS
  C4 liver pathways in CSV: 26
  Our 26 defined: 26
  ✅ C4 pathway names MATCH our 26 definitions exactly

  26 original pathways → 153 unique genes
  29 pathways (with 3 new) → 179 unique genes
  Genes in ≥2 pathways: 33
  Multi-pathway genes: ['BCL2', 'BCL6', 'CCNB1', 'CCND1', 'CDK1', 'CTLA4', 'EOMES', 'GNLY', 'GZMB', 'GZMK', 'HAVCR2', 'IL2RB', 'IL2RG', 'IL7R', 'KLRC2', 'KLRD1', 'LAG3', 'LEF1', 'MKI67', 'PCNA', 'PDCD1', 'PRDM1', 'PRF1', 'RB1', 'SELL', 'TCF7', 'TGFB1', 'TGFB2', 'TGFBR1', 'TGFBR2', 'TIGIT', 'TOP2A', 'TOX']

✅ Cell 4 complete


In [ ]:
# =================================================================
# CELL 5: CROSS-VERIFICATION — C3 vs C5 overlap
# =================================================================
print("=" * 70)
print("  CROSS-VERIFICATION: C3 ∩ C5")
print("=" * 70)

if C3_GENES and C5_GENES:
    c3_set = set(C3_GENES)
    c5_set = set(C5_GENES)
    shared = c3_set & c5_set
    c3_only = c3_set - c5_set
    c5_only = c5_set - c3_set

    print(f"  C3: {len(c3_set)} genes")
    print(f"  C5: {len(c5_set)} genes")
    print(f"  Shared (C3∩C5): {len(shared)}")
    print(f"  C3-only: {len(c3_only)}")
    print(f"  C5-only: {len(c5_only)}")

    # Expected from manuscript: C3=196, C5=148, Shared=102
    print(f"\n  === MANUSCRIPT CLAIM CHECK ===")
    print(f"  C3 = {len(c3_set)} (expected 196) → {'✅' if len(c3_set)==196 else '❌ MISMATCH'}")
    print(f"  C5 = {len(c5_set)} (expected 148) → {'✅' if len(c5_set)==148 else '❌ MISMATCH'}")
    print(f"  Shared = {len(shared)} (expected 102) → {'✅' if len(shared)==102 else '⚠️ CHECK'}")

    # C9B genes check
    C9B = ['MX1','ISG15','STAT2','IRF3','IRF7','SOCS1','SOCS3','AICDA','JCHAIN',
           'HLA-DRA','HLA-DRB1','HLA-DPB1','HLA-DPA1','CD74','B2M','TAP1',
           'IL1RN','STAT1','IL1B']
    c9b_in_c3 = [g for g in C9B if g in c3_set]
    c9b_not_in_c3 = [g for g in C9B if g not in c3_set]
    print(f"\n  C9B genes in C3: {len(c9b_in_c3)}/19")
    if c9b_not_in_c3:
        print(f"  ⚠️ C9B genes NOT in C3: {c9b_not_in_c3}")

    # Key C3-only genes for manuscript
    key_c3_only = ['MX1','ISG15','STAT2','IRF3','IRF7','SOCS1','SOCS3',
                   'AICDA','JCHAIN','HLA-DRA','HLA-DRB1','HLA-DPB1','HLA-DPA1',
                   'CD74','TAP1','IL1RN','STAT1','IL1B','LDHA']
    print(f"\n  Key manuscript genes — C3/C5 status:")
    for g in key_c3_only:
        in3 = '✅' if g in c3_set else '❌'
        in5 = '✅' if g in c5_set else '❌'
        print(f"    {g:<12s}  C3:{in3}  C5:{in5}")
else:
    print("  🔴 Cannot verify — gene lists not loaded")

print(f"\n✅ Cell 5 complete")

  CROSS-VERIFICATION: C3 ∩ C5
  C3: 196 genes
  C5: 148 genes
  Shared (C3∩C5): 81
  C3-only: 115
  C5-only: 67

  === MANUSCRIPT CLAIM CHECK ===
  C3 = 196 (expected 196) → ✅
  C5 = 148 (expected 148) → ✅
  Shared = 81 (expected 102) → ⚠️ CHECK

  C9B genes in C3: 19/19

  Key manuscript genes — C3/C5 status:
    MX1           C3:✅  C5:❌
    ISG15         C3:✅  C5:❌
    STAT2         C3:✅  C5:❌
    IRF3          C3:✅  C5:❌
    IRF7          C3:✅  C5:❌
    SOCS1         C3:✅  C5:❌
    SOCS3         C3:✅  C5:❌
    AICDA         C3:✅  C5:❌
    JCHAIN        C3:✅  C5:❌
    HLA-DRA       C3:✅  C5:❌
    HLA-DRB1      C3:✅  C5:❌
    HLA-DPB1      C3:✅  C5:❌
    HLA-DPA1      C3:✅  C5:❌
    CD74          C3:✅  C5:❌
    TAP1          C3:✅  C5:❌
    IL1RN         C3:✅  C5:❌
    STAT1         C3:✅  C5:❌
    IL1B          C3:✅  C5:❌
    LDHA          C3:✅  C5:❌

✅ Cell 5 complete


In [ ]:
!pip install scanpy anndata matplotlib seaborn scipy -q

import subprocess
import sys

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 55.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 176.6/176.6 kB 22.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.1/60.1 kB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 284.1/284.1 kB 34.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 140.1 MB/s eta 0:00:00


In [ ]:
# =================================================================
# CELL 6: PATHWAY GENE UNIVERSE — Verify 182 unique genes
# =================================================================
print("=" * 70)
print("  PATHWAY GENE UNIVERSE VERIFICATION")
print("=" * 70)

# Check against h5ad to find which genes are actually present
DATA_PATH = '/content/drive/MyDrive/ITLAS/data/processed/GSE182159_gut2021_annotated.h5ad'
try:
    import scanpy as sc
    adata = sc.read_h5ad(DATA_PATH, backed='r')
    h5ad_genes = set(adata.var_names)
    print(f"  h5ad gene universe: {len(h5ad_genes)} genes")

    # Check each pathway
    print(f"\n  {'Pathway':<25s} {'Defined':>7s} {'Found':>5s} {'Missing':>7s}")
    print("  " + "-" * 50)

    total_defined = 0
    total_found = 0
    all_found_genes = set()
    all_missing_genes = set()

    for pw in sorted(ALL_29.keys()):
        genes = ALL_29[pw]
        found = [g for g in genes if g in h5ad_genes]
        missing = [g for g in genes if g not in h5ad_genes]
        total_defined += len(genes)
        total_found += len(found)
        all_found_genes.update(found)
        all_missing_genes.update(missing)

        status = '✅' if not missing else '⚠️'
        print(f"  {status} {pw:<23s} {len(genes):>7d} {len(found):>5d} {len(missing):>7d}", end='')
        if missing:
            print(f"  → {missing}", end='')
        print()

    print(f"\n  SUMMARY:")
    print(f"  Total gene mentions (with overlap): {total_defined}")
    print(f"  Unique genes across 29 pathways: {len(all_pw_genes_29)}")
    print(f"  Found in h5ad: {len(all_found_genes)}")
    print(f"  Missing from h5ad: {len(all_missing_genes)} → {sorted(all_missing_genes)}")
    print(f"\n  Manuscript claims 182 unique genes → actual: {len(all_found_genes)}")
    print(f"  {'✅ MATCH' if len(all_found_genes)==182 else '⚠️ DISCREPANCY: ' + str(len(all_found_genes))}")

    adata.file.close()

except Exception as e:
    print(f"  ⚠️ Cannot load h5ad: {e}")
    print(f"  Using definition-only count: {len(all_pw_genes_29)} unique genes")
    print(f"  Manuscript claims 182 → {'✅' if len(all_pw_genes_29)==182 else '⚠️ ' + str(len(all_pw_genes_29))}")

print(f"\n✅ Cell 6 complete")

  PATHWAY GENE UNIVERSE VERIFICATION
  h5ad gene universe: 24452 genes

  Pathway                   Defined Found Missing
  --------------------------------------------------
  ✅ angiogenesis                  9     9       0
  ✅ antigen_presentation         10    10       0
  ✅ apoptosis                     6     6       0
  ✅ cancer_associated             9     9       0
  ✅ cell_cycle                   10    10       0
  ✅ checkpoint                    6     6       0
  ✅ cytotoxicity                  7     7       0
  ✅ epigenetics                   8     8       0
  ✅ exhaustion                    6     6       0
  ✅ fibrosis                     10    10       0
  ✅ glycolysis                    8     8       0
  ⚠️ il15_mtor                    10     9       1  → ['RPS6KB1']
  ✅ immune_evasion                8     8       0
  ✅ inflammasome                  6     6       0
  ✅ memory_t                      8     8       0
  ✅ metabolic_recovery            6     6       0
  ✅ mito_

In [ ]:
# =================================================================
# CELL 7: GENERATE CORRECTED S1/S2 (if discrepancies found)
# =================================================================
print("=" * 70)
print("  FINAL REPORT — S1/S2 VERIFICATION SUMMARY")
print("=" * 70)

print(f"""
  S1 (29 Pathway Definitions):
    Pathways: {len(ALL_29)} (26 original + 3 new)
    Unique genes (definition): {len(all_pw_genes_29)}
    Multi-pathway genes: {len(multi)}
    Expected in manuscript: 29 pathways, 182 unique genes

  S2 (Gene Candidates):
    C3: {len(C3_GENES)} genes (expected 196)
    C5: {len(C5_GENES)} genes (expected 148)
    Shared: {len(set(C3_GENES) & set(C5_GENES)) if C3_GENES and C5_GENES else 'N/A'}
    C3-only: {len(set(C3_GENES) - set(C5_GENES)) if C3_GENES and C5_GENES else 'N/A'}
    C5-only: {len(set(C5_GENES) - set(C3_GENES)) if C3_GENES and C5_GENES else 'N/A'}
    C9B: 19 genes
    Expected in manuscript: 196 / 148 / 102 shared
""")

# If any mismatch, output corrected gene lists for S2 regeneration
if C3_GENES and len(C3_GENES) != 196:
    print(f"  🔴 C3 MISMATCH: {len(C3_GENES)} vs 196")
    print(f"     Actual C3 genes saved to: C3_verified_genes.txt")
    with open(f'{BASE}/C3_verified_genes.txt', 'w') as f:
        for g in sorted(C3_GENES):
            f.write(g + '\n')

if C5_GENES and len(C5_GENES) != 148:
    print(f"  🔴 C5 MISMATCH: {len(C5_GENES)} vs 148")
    print(f"     Actual C5 genes saved to: C5_verified_genes.txt")
    with open(f'{BASE}/C5_verified_genes.txt', 'w') as f:
        for g in sorted(C5_GENES):
            f.write(g + '\n')

# Save full verification report
report = {
    'C3_count': len(C3_GENES),
    'C5_count': len(C5_GENES),
    'shared_count': len(set(C3_GENES) & set(C5_GENES)) if C3_GENES and C5_GENES else None,
    'pathway_unique_genes': len(all_pw_genes_29),
    'multi_pathway_genes': len(multi),
    'C3_genes': sorted(C3_GENES),
    'C5_genes': sorted(C5_GENES),
}
with open(f'{BASE}/S1_S2_verification_report.json', 'w') as f:
    json.dump(report, f, indent=2)

print(f"\n  Report saved: {BASE}/S1_S2_verification_report.json")
print(f"\n{'=' * 70}")
print(f"  ✅ S1/S2 VERIFICATION COMPLETE")
print(f"  If all ✅ → S1/S2 are correct as generated")
print(f"  If any ❌ → use verified gene lists to regenerate S1/S2")
print(f"{'=' * 70}")

  FINAL REPORT — S1/S2 VERIFICATION SUMMARY

  S1 (29 Pathway Definitions):
    Pathways: 29 (26 original + 3 new)
    Unique genes (definition): 179
    Multi-pathway genes: 33
    Expected in manuscript: 29 pathways, 182 unique genes
    
  S2 (Gene Candidates):
    C3: 196 genes (expected 196)
    C5: 148 genes (expected 148)
    Shared: 81
    C3-only: 115
    C5-only: 67
    C9B: 19 genes
    Expected in manuscript: 196 / 148 / 102 shared


  Report saved: /content/drive/MyDrive/ITLAS/results/version18-analysis/S1_S2_verification_report.json

  ✅ S1/S2 VERIFICATION COMPLETE
  If all ✅ → S1/S2 are correct as generated
  If any ❌ → use verified gene lists to regenerate S1/S2


# V18 Supplementary Table S1/S2 Generation
**Run AFTER the verification notebook (Cells 1-7).** Variables , , , , , , ,  must be in memory.

Cells 8-11 generate and verify S1/S2 xlsx files, saved directly to Google Drive.

In [ ]:
# =================================================================
# CELL 8: INSTALL OPENPYXL + DEFINE STYLES
# =================================================================
!pip install openpyxl -q

import openpyxl
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side

OUT_DIR = '/content/drive/MyDrive/ITLAS/results/version18-analysis'

# Styles
header_font = Font(name='Arial', bold=True, size=10, color='FFFFFF')
header_fill = PatternFill('solid', fgColor='2F5496')
cat_fill = PatternFill('solid', fgColor='D6E4F0')
data_font = Font(name='Arial', size=10)
title_font = Font(name='Arial', bold=True, size=12)
thin_border = Border(
    left=Side(style='thin', color='CCCCCC'),
    right=Side(style='thin', color='CCCCCC'),
    top=Side(style='thin', color='CCCCCC'),
    bottom=Side(style='thin', color='CCCCCC')
)
green_fill = PatternFill('solid', fgColor='E2EFDA')
orange_fill = PatternFill('solid', fgColor='FCE4D6')

def style_header(ws, row, ncols):
    for c in range(1, ncols+1):
        cell = ws.cell(row=row, column=c)
        cell.font = header_font
        cell.fill = header_fill
        cell.alignment = Alignment(horizontal='center', vertical='center', wrap_text=True)
        cell.border = thin_border

def style_data(ws, row, ncols, fill=None):
    for c in range(1, ncols+1):
        cell = ws.cell(row=row, column=c)
        cell.font = data_font
        cell.border = thin_border
        if fill:
            cell.fill = fill

# Verify prerequisites
c3_set = set(C3_GENES)
c5_set = set(C5_GENES)
shared = c3_set & c5_set
print(f"Prerequisites check:")
print(f"  C3: {len(c3_set)}, C5: {len(c5_set)}, Shared: {len(shared)}")
print(f"  Pathways: {len(ALL_29)}")
print(f"  Unique pathway genes: {len(all_pw_genes_29)}")
print(f"✅ Cell 8 ready")

Prerequisites check:
  C3: 196, C5: 148, Shared: 81
  Pathways: 29
  Unique pathway genes: 179
✅ Cell 8 ready


In [ ]:
# =================================================================
# CELL 9: GENERATE TABLE S1 (29 Pathway Definitions)
# =================================================================
print("=" * 70)
print("  GENERATING SUPPLEMENTARY TABLE S1")
print("=" * 70)

CATEGORIES = {
    'inflammasome': 'Immune Effector', 'cytotoxicity': 'Immune Effector',
    'checkpoint': 'Immune Effector', 'exhaustion': 'Immune Effector',
    'nk_function': 'Immune Effector',
    'nk_il15_dual': 'Immune Regulation', 'il15_mtor': 'Immune Regulation',
    'immune_evasion': 'Immune Regulation', 'treg': 'Immune Regulation',
    'naive_t': 'Immune Regulation',
    'memory_t': 'T Cell Biology', 'tf_programs': 'T Cell Biology',
    'tissue_resident': 'T Cell Biology', 'stemness': 'T Cell Biology',
    'glycolysis': 'Metabolism', 'oxphos': 'Metabolism',
    'mito_dysfunction': 'Metabolism', 'metabolic_recovery': 'Metabolism',
    'cancer_associated': 'Disease Progression', 'fibrosis': 'Disease Progression',
    'senescence': 'Disease Progression', 'epigenetics': 'Disease Progression',
    'angiogenesis': 'Disease Progression', 'cell_cycle': 'Disease Progression',
    'proliferation': 'Disease Progression', 'apoptosis': 'Disease Progression',
    'antigen_presentation': 'Antigen Processing', 'type1_ifn': 'Interferon Response',
    'tgfb_signaling': 'TGF-beta Signaling',
}

pathway_order = [
    'inflammasome', 'cytotoxicity', 'checkpoint', 'exhaustion', 'nk_function',
    'nk_il15_dual', 'il15_mtor', 'immune_evasion', 'treg', 'naive_t',
    'memory_t', 'tf_programs', 'tissue_resident', 'stemness',
    'glycolysis', 'oxphos', 'mito_dysfunction', 'metabolic_recovery',
    'cancer_associated', 'fibrosis', 'senescence', 'epigenetics',
    'angiogenesis', 'cell_cycle', 'proliferation', 'apoptosis',
    'antigen_presentation', 'type1_ifn', 'tgfb_signaling'
]

# Count genes found in h5ad per pathway
try:
    h5ad_genes_set = set(adata.var_names)
except:
    h5ad_genes_set = None
    print("  ⚠️ h5ad not in memory, skipping h5ad check column")

wb1 = openpyxl.Workbook()
ws1 = wb1.active
ws1.title = "S1_Pathway_Definitions"

# Title
ws1.cell(row=1, column=1, value="Supplementary Table S1. Gene Set Definitions for 29 Immunological Pathways Used in AUCell Pathway Scoring (C2/C4)")
ws1.cell(row=1, column=1).font = title_font
ws1.merge_cells('A1:G1')

n_unique = len(all_pw_genes_29)
n_found = len(all_found_genes) if 'all_found_genes' in dir() else n_unique
n_missing_genes = len(all_missing_genes) if 'all_missing_genes' in dir() else 0
ws1.cell(row=2, column=1, value=f"ITLAS v18 tissue-separated reanalysis of GSE182159. 26 original pathways (C2) + 3 added in C9 (marked *). {n_unique} unique genes defined; {n_found} found in h5ad ({n_missing_genes} absent: {', '.join(sorted(all_missing_genes)) if 'all_missing_genes' in dir() and all_missing_genes else 'none'}).")
ws1.cell(row=2, column=1).font = Font(name='Arial', size=9, italic=True)
ws1.merge_cells('A2:G2')

# Headers
headers = ['Category', 'Pathway Name', 'N Genes (Defined)', 'N Genes (in h5ad)', 'Gene List', 'Source', 'Shared with Other Pathways']
for i, h in enumerate(headers, 1):
    ws1.cell(row=4, column=i, value=h)
style_header(ws1, 4, 7)

row = 5
current_cat = None
for pw in pathway_order:
    genes = ALL_29[pw]
    cat = CATEGORIES[pw]
    is_new = pw in GENE_SETS_3NEW

    fill = None
    if cat != current_cat:
        current_cat = cat
        fill = cat_fill

    pw_display = f"{pw}*" if is_new else pw
    source = "C9 (added)" if is_new else "C2 (original)"

    # h5ad availability
    if h5ad_genes_set:
        found = [g for g in genes if g in h5ad_genes_set]
        missing = [g for g in genes if g not in h5ad_genes_set]
        n_found_pw = len(found)
        gene_display = ', '.join(found)
        if missing:
            gene_display += f' [ABSENT: {", ".join(missing)}]'
    else:
        n_found_pw = len(genes)
        gene_display = ', '.join(genes)

    # Shared genes
    shared_with = []
    for other_pw, other_genes in ALL_29.items():
        if other_pw != pw:
            overlap = set(genes) & set(other_genes)
            if overlap:
                for g in overlap:
                    shared_with.append(f"{g}({other_pw})")
    shared_str = '; '.join(sorted(set(shared_with)))[:250] if shared_with else ""

    ws1.cell(row=row, column=1, value=cat)
    ws1.cell(row=row, column=2, value=pw_display)
    ws1.cell(row=row, column=3, value=len(genes))
    ws1.cell(row=row, column=4, value=n_found_pw)
    ws1.cell(row=row, column=5, value=gene_display)
    ws1.cell(row=row, column=6, value=source)
    ws1.cell(row=row, column=7, value=shared_str)
    style_data(ws1, row, 7, fill=fill)
    ws1.cell(row=row, column=3).alignment = Alignment(horizontal='center')
    ws1.cell(row=row, column=4).alignment = Alignment(horizontal='center')
    row += 1

# Summary
row += 1
ws1.cell(row=row, column=1, value="TOTAL")
ws1.cell(row=row, column=2, value=f"{len(ALL_29)} pathways")
ws1.cell(row=row, column=3, value=n_unique)
ws1.cell(row=row, column=4, value=n_found)
ws1.cell(row=row, column=5, value=f"{n_unique} unique genes defined; {len(multi)} genes shared across ≥2 pathways")
for c in range(1, 8):
    ws1.cell(row=row, column=c).border = thin_border
    ws1.cell(row=row, column=c).font = Font(name='Arial', bold=True, size=10)

# Column widths
ws1.column_dimensions['A'].width = 20
ws1.column_dimensions['B'].width = 24
ws1.column_dimensions['C'].width = 14
ws1.column_dimensions['D'].width = 14
ws1.column_dimensions['E'].width = 80
ws1.column_dimensions['F'].width = 14
ws1.column_dimensions['G'].width = 60

s1_path = f'{OUT_DIR}/Supplementary_Table_S1.xlsx'
wb1.save(s1_path)
print(f"  ✅ S1 saved: {s1_path}")
print(f"     {len(ALL_29)} pathways, {n_unique} unique genes, {n_found} in h5ad")

  GENERATING SUPPLEMENTARY TABLE S1
  ✅ S1 saved: /content/drive/MyDrive/ITLAS/results/version18-analysis/Supplementary_Table_S1.xlsx
     29 pathways, 179 unique genes, 178 in h5ad


In [ ]:
# =================================================================
# CELL 10: GENERATE TABLE S2 (196 Gene Candidates)
# =================================================================
print("=" * 70)
print("  GENERATING SUPPLEMENTARY TABLE S2")
print("=" * 70)

# C9B genes
C9B_GENES = [
    'MX1', 'ISG15', 'STAT2', 'IRF3', 'IRF7', 'SOCS1', 'SOCS3', 'AICDA', 'JCHAIN',
    'HLA-DRA', 'HLA-DRB1', 'HLA-DPB1', 'HLA-DPA1', 'CD74', 'B2M', 'TAP1',
    'IL1RN', 'STAT1', 'IL1B'
]

# Key manuscript roles
KEY_ROLES = {
    'MX1': 'IFN response, IT-specific (FDR Tier 1)',
    'ISG15': 'IFN response, IT-specific (FDR Tier 1)',
    'STAT2': 'IFN response, IT-specific (FDR Tier 1)',
    'IRF3': 'IFN response, IT-specific (FDR Tier 1)',
    'IRF7': 'IFN response (FDR Tier 1)',
    'SOCS1': 'JAK-STAT brake, IT-specific (FDR Tier 1)',
    'SOCS3': 'JAK-STAT brake, IT-specific (FDR Tier 1)',
    'DNMT1': 'Epigenetic silencing, pan-tissue (FDR Tier 1)',
    'DNMT3A': 'Epigenetic silencing (FDR Tier 1)',
    'TGFB1': 'Paracrine suppression (FDR Tier 1)',
    'LGALS9': 'Paracrine suppression (FDR Tier 1)',
    'MTOR': 'Metabolic checkpoint (FDR Tier 1)',
    'JAK1': 'Most frequent sig gene (32 tests)',
    'TOX': 'Liver-specific exhaustion (Tier 2)',
    'PRDM1': 'Differentiation block, pan-tissue (Tier 2)',
    'HLA-DRA': 'Antigen presentation (FDR Tier 1)',
    'HLA-DRB1': 'Antigen presentation (FDR Tier 1)',
    'HLA-DPB1': 'Antigen presentation (FDR Tier 1)',
    'HLA-DPA1': 'Antigen presentation (FDR Tier 1)',
    'CD74': 'Antigen presentation (FDR Tier 1)',
    'AICDA': 'B cell CSR (Tier 3, sparse)',
    'JCHAIN': 'B cell phenotype (FDR Tier 1)',
    'IL2RA': 'B cell activation',
    'TYROBP': 'PlasmaB cytotoxic phenotype',
    'TFAM': 'Mito biogenesis, pan-immune',
    'TGFBR2': 'Pan-immune, rho=0.88 with JAK1',
    'EIF4EBP1': 'Metabolic dissociation (Liver NK)',
    'LDHA': 'Metabolic checkpoint (MIXED attribution)',
    'BAK1': 'Myeloid apoptosis, pan-tissue',
    'STAT1': 'IFN signaling (FDR Tier 1)',
    'IL1RN': 'Tolerance brake',
    'TAP1': 'Antigen processing (FDR Tier 1)',
    'SERPINE1': 'IT-to-IA NK transition',
    'ID3': 'IT-to-IA NK transition',
    'LAYN': 'Liver exhaustion (tissue-opposite)',
    'BCL6': 'Liver CD8T stemness',
    'RORC': 'Liver CD4T Treg skewing',
    'CTLA4': 'Liver CD4T Treg (tissue-opposite)',
    'TIGIT': 'Liver T cell checkpoint',
    'FOXP3': 'IA-AR discriminator',
    'MT-CYB': 'Mito co-regulation hub (rho=0.93)',
    'MEFV': 'Inflammasome sensor, CR scar',
    'B2M': 'Antigen processing',
    'IL1B': 'Inflammasome effector',
    'TET2': 'Epigenetic regulator',
}

# Also load C3b info if available
c3b_genes = set()
try:
    c3b_mask = df_c3_genes['in_C3b'] == True
    c3b_genes = set(df_c3_genes.loc[c3b_mask, 'gene'].tolist())
    c3_strict = set(df_c3_genes.loc[df_c3_genes['in_C3'] == True, 'gene'].tolist())
    print(f"  C3 strict (in_C3=True): {len(c3_strict)}")
    print(f"  C3b (in_C3b=True): {len(c3b_genes)}")
    print(f"  C3b-only (not in C3 strict): {len(c3b_genes - c3_strict)}")
except:
    c3_strict = c3_set
    print("  ⚠️ No C3b column info, using flat C3 list")

# Full gene universe = C3 ∪ C5
all_genes = sorted(c3_set | c5_set)
print(f"  Total gene universe (C3∪C5): {len(all_genes)}")

wb2 = openpyxl.Workbook()
ws2 = wb2.active
ws2.title = "S2_Gene_Candidates"

# Title
ws2.cell(row=1, column=1, value="Supplementary Table S2. Individual Gene Candidates with Analytical Component Source Attribution")
ws2.cell(row=1, column=1).font = title_font
ws2.merge_cells('A1:I1')

ws2.cell(row=2, column=1, value=f"C3: {len(c3_set)} genes (broad screen); C5: {len(c5_set)} genes (refined panel); C9B: {len(C9B_GENES)} genes (re-verified with stratified FDR). {len(shared)} genes shared between C3 and C5.")
ws2.cell(row=2, column=1).font = Font(name='Arial', size=9, italic=True)
ws2.merge_cells('A2:I2')

# Headers
headers2 = ['Gene', 'C3 (196)', 'C3b', 'C5 (148)', 'C9B (19)', 'Source Category', 'Pathway Membership', 'N Pathways', 'Key Manuscript Role']
for i, h in enumerate(headers2, 1):
    ws2.cell(row=4, column=i, value=h)
style_header(ws2, 4, 9)

row = 5
for gene in all_genes:
    in_c3 = gene in c3_set
    in_c5 = gene in c5_set
    in_c9b = gene in C9B_GENES
    in_c3b = gene in c3b_genes

    # Source category
    if in_c3 and in_c5:
        src = "C3+C5 (shared)"
    elif in_c3 and not in_c5:
        src = "C3-only"
    elif in_c5 and not in_c3:
        src = "C5-only"
    else:
        src = "Other"

    # Pathway membership
    pw_list = []
    for pw, genes in ALL_29.items():
        if gene in genes:
            pw_list.append(pw)
    pw_str = ', '.join(pw_list) if pw_list else 'None (literature-derived)'

    ws2.cell(row=row, column=1, value=gene)
    ws2.cell(row=row, column=2, value="Yes" if in_c3 else "No")
    ws2.cell(row=row, column=3, value="Yes" if in_c3b else "")
    ws2.cell(row=row, column=4, value="Yes" if in_c5 else "No")
    ws2.cell(row=row, column=5, value="Yes" if in_c9b else "")
    ws2.cell(row=row, column=6, value=src)
    ws2.cell(row=row, column=7, value=pw_str)
    ws2.cell(row=row, column=8, value=len(pw_list) if pw_list else 0)
    ws2.cell(row=row, column=9, value=KEY_ROLES.get(gene, ''))

    # Color coding
    fill = None
    if in_c9b:
        fill = green_fill
    elif src == "C3-only":
        fill = orange_fill
    style_data(ws2, row, 9, fill=fill)

    for c in [2, 3, 4, 5, 8]:
        ws2.cell(row=row, column=c).alignment = Alignment(horizontal='center')
    row += 1

# Summary rows
row += 1
summaries = [
    ("C3 total", len(c3_set)),
    ("C5 total", len(c5_set)),
    ("C3 ∩ C5 shared", len(shared)),
    ("C3-only", len(c3_set - c5_set)),
    ("C5-only", len(c5_set - c3_set)),
    ("C9B re-verified", len(C9B_GENES)),
    ("Gene universe (C3∪C5)", len(all_genes)),
]
for label, val in summaries:
    ws2.cell(row=row, column=1, value=label)
    ws2.cell(row=row, column=2, value=str(val))
    ws2.cell(row=row, column=1).font = Font(name='Arial', bold=True, size=10)
    ws2.cell(row=row, column=2).font = data_font
    for c in range(1, 10):
        ws2.cell(row=row, column=c).border = thin_border
    row += 1

# Column widths
ws2.column_dimensions['A'].width = 14
ws2.column_dimensions['B'].width = 10
ws2.column_dimensions['C'].width = 8
ws2.column_dimensions['D'].width = 10
ws2.column_dimensions['E'].width = 10
ws2.column_dimensions['F'].width = 18
ws2.column_dimensions['G'].width = 55
ws2.column_dimensions['H'].width = 12
ws2.column_dimensions['I'].width = 42

# Legend sheet
ws_leg = wb2.create_sheet("Legend")
ws_leg.cell(row=1, column=1, value="Color Legend & Definitions")
ws_leg.cell(row=1, column=1).font = Font(name='Arial', bold=True, size=11)
legends = [
    ("Green", "C9B re-verified gene (stratified FDR applied)", 'E2EFDA'),
    ("Orange", "C3-only gene (not in C5 148-gene panel)", 'FCE4D6'),
    ("No color", "C3+C5 shared or C5-only gene", None),
]
for i, (label, desc, color) in enumerate(legends, 3):
    ws_leg.cell(row=i, column=1, value=label)
    ws_leg.cell(row=i, column=2, value=desc)
    if color:
        ws_leg.cell(row=i, column=1).fill = PatternFill('solid', fgColor=color)

ws_leg.cell(row=7, column=1, value="Column Definitions")
ws_leg.cell(row=7, column=1).font = Font(name='Arial', bold=True, size=11)
defs = [
    ("C3 (196)", "Broad individual gene expression screen (196 candidates)"),
    ("C3b", "Genes added via C3b supplementary analysis (e.g., AICDA, JCHAIN)"),
    ("C5 (148)", "Refined gene panel after expression filtering"),
    ("C9B (19)", "Critical C3-only genes re-verified with stratified FDR in C9B"),
    ("Source Category", "C3+C5 shared / C3-only / C5-only"),
    ("Pathway Membership", "Which of the 29 pathways this gene belongs to (if any)"),
    ("Key Manuscript Role", "Role of this gene in manuscript findings (blank = not a key finding)"),
]
for i, (col, desc) in enumerate(defs, 8):
    ws_leg.cell(row=i, column=1, value=col)
    ws_leg.cell(row=i, column=2, value=desc)
    ws_leg.cell(row=i, column=1).font = Font(name='Arial', bold=True, size=10)

ws_leg.column_dimensions['A'].width = 22
ws_leg.column_dimensions['B'].width = 70

s2_path = f'{OUT_DIR}/Supplementary_Table_S2.xlsx'
wb2.save(s2_path)
print(f"  ✅ S2 saved: {s2_path}")
print(f"     {len(all_genes)} genes, C3={len(c3_set)}, C5={len(c5_set)}, Shared={len(shared)}")

  GENERATING SUPPLEMENTARY TABLE S2
  C3 strict (in_C3=True): 191
  C3b (in_C3b=True): 17
  C3b-only (not in C3 strict): 5
  Total gene universe (C3∪C5): 263
  ✅ S2 saved: /content/drive/MyDrive/ITLAS/results/version18-analysis/Supplementary_Table_S2.xlsx
     263 genes, C3=196, C5=148, Shared=81


In [ ]:
# =================================================================
# CELL 11: FINAL VERIFICATION OF GENERATED FILES
# =================================================================
print("=" * 70)
print("  FINAL VERIFICATION OF GENERATED S1/S2")
print("=" * 70)

# Re-read and verify
wb1_check = openpyxl.load_workbook(s1_path)
ws1_check = wb1_check.active
pw_count = 0
for r in range(5, ws1_check.max_row):
    if ws1_check.cell(r, 2).value and ws1_check.cell(r, 1).value != "TOTAL":
        pw_count += 1
print(f"  S1: {pw_count} pathways listed")

wb2_check = openpyxl.load_workbook(s2_path)
ws2_check = wb2_check.active
c3_yes = sum(1 for r in range(5, ws2_check.max_row) if ws2_check.cell(r, 2).value == 'Yes')
c5_yes = sum(1 for r in range(5, ws2_check.max_row) if ws2_check.cell(r, 4).value == 'Yes')
c9b_yes = sum(1 for r in range(5, ws2_check.max_row) if ws2_check.cell(r, 5).value == 'Yes')
print(f"  S2: C3=Yes:{c3_yes}, C5=Yes:{c5_yes}, C9B=Yes:{c9b_yes}")

# Cross-check against manuscript numbers
print(f"\n  === MANUSCRIPT NUMBER VERIFICATION ===")
checks = [
    ("29 pathways (S1)", pw_count, 29),
    (f"Unique pathway genes", n_unique, 179),
    ("C3 genes (S2)", c3_yes, 196),
    ("C5 genes (S2)", c5_yes, 148),
    ("C3∩C5 shared", len(shared), 81),
    ("C9B genes", c9b_yes, 19),
]
all_pass = True
for label, actual, expected in checks:
    ok = actual == expected
    if not ok:
        all_pass = False
    print(f"  {'✅' if ok else '❌'} {label}: {actual} (expected {expected})")

print(f"\n  === NUMBERS TO UPDATE IN MANUSCRIPT M&M ===")
print(f"  '182 unique genes' → {n_unique}")
print(f"  '102 genes shared' → {len(shared)}")
print(f"  All other numbers (196, 148, 29, 19) → ✅ confirmed correct")

if all_pass:
    print(f"\n  🎉 ALL CHECKS PASSED — S1/S2 are publication-ready")
else:
    print(f"\n  ⚠️ Some checks failed — review above")

print(f"\n  Files saved:")
print(f"    {s1_path}")
print(f"    {s2_path}")
print(f"\n{'=' * 70}")
print(f"  ✅ S1/S2 GENERATION COMPLETE")
print(f"{'=' * 70}")

  FINAL VERIFICATION OF GENERATED S1/S2
  S1: 29 pathways listed
  S2: C3=Yes:196, C5=Yes:148, C9B=Yes:19

  === MANUSCRIPT NUMBER VERIFICATION ===
  ✅ 29 pathways (S1): 29 (expected 29)
  ✅ Unique pathway genes: 179 (expected 179)
  ✅ C3 genes (S2): 196 (expected 196)
  ✅ C5 genes (S2): 148 (expected 148)
  ✅ C3∩C5 shared: 81 (expected 81)
  ✅ C9B genes: 19 (expected 19)

  === NUMBERS TO UPDATE IN MANUSCRIPT M&M ===
  '182 unique genes' → 179
  '102 genes shared' → 81
  All other numbers (196, 148, 29, 19) → ✅ confirmed correct

  🎉 ALL CHECKS PASSED — S1/S2 are publication-ready

  Files saved:
    /content/drive/MyDrive/ITLAS/results/version18-analysis/Supplementary_Table_S1.xlsx
    /content/drive/MyDrive/ITLAS/results/version18-analysis/Supplementary_Table_S2.xlsx

  ✅ S1/S2 GENERATION COMPLETE


In [8]:
######################
# Result 2.1. Identification of IT-Specific Genes 에서,
# Suppleementary Table S2 and text content 사이에
# liver and blood sample 숫자 불일치 문제 해결 및
# Table S2 에 sheet-2 추가함
######################

In [2]:
# ========================================
# C8 IT-specific data 확인
# ========================================
import os, glob

C8_DIR = '/content/drive/MyDrive/ITLAS/results/version18-analysis/C8_characteristics'
print("C8 files:")
if os.path.exists(C8_DIR):
    for f in sorted(os.listdir(C8_DIR)):
        fpath = os.path.join(C8_DIR, f)
        size = os.path.getsize(fpath)
        print(f"  {f} ({size:,} bytes)")
        if f.endswith('.csv'):
            df = pd.read_csv(fpath)
            print(f"    Shape: {df.shape}")
            print(f"    Columns: {list(df.columns)[:10]}")
            # Check for IT-specific
            for col in df.columns:
                if 'pattern' in col.lower() or 'class' in col.lower() or 'type' in col.lower():
                    vals = df[col].value_counts()
                    print(f"    {col}: {dict(vals)}")
else:
    print("  C8_DIR not found!")

# Also check C5 directory for pattern classification
C5_DIR = '/content/drive/MyDrive/ITLAS/results/version18-analysis/C5_genes'
print("\nC5 files:")
if os.path.exists(C5_DIR):
    for f in sorted(os.listdir(C5_DIR)):
        fpath = os.path.join(C5_DIR, f)
        size = os.path.getsize(fpath)
        print(f"  {f} ({size:,} bytes)")
        if f.endswith('.csv') and 'pattern' in f.lower():
            df = pd.read_csv(fpath)
            print(f"    Shape: {df.shape}")
            print(f"    Columns: {list(df.columns)}")

C8 files:
  C8_CR_scar_genes.csv (6,472 bytes)
    Shape: (114, 7)
    Columns: ['tissue', 'lineage', 'gene', 'pathway', 'CR_pct', 'CR_p', 'IA_p']
  C8_IA_AR_discriminators.csv (5,059 bytes)
    Shape: (48, 17)
    Columns: ['tissue', 'gene', 'lineage', 'pathway', 'comparison', 'stage1', 'stage2', 'mean_s1', 'mean_s2', 'pct_change']
  C8_IA_transition_genes.csv (5,482 bytes)
    Shape: (54, 17)
    Columns: ['tissue', 'gene', 'lineage', 'pathway', 'comparison', 'stage1', 'stage2', 'mean_s1', 'mean_s2', 'pct_change']
  C8_IT_specific_genes.csv (6,885 bytes)
    Shape: (122, 7)
    Columns: ['tissue', 'lineage', 'gene', 'pathway', 'IT_pct', 'IT_p', 'IA_p']

C5 files:
  C5_all_significant_genes.csv (104,015 bytes)
  C5_gene_significance_ranking.csv (3,321 bytes)
  C5_genes_blood.csv (708,365 bytes)
  C5_genes_liver.csv (708,493 bytes)


In [3]:
# ========================================
# C8 IT-specific: 내용 분석
# ========================================
import pandas as pd

it_spec = pd.read_csv('/content/drive/MyDrive/ITLAS/results/version18-analysis/C8_characteristics/C8_IT_specific_genes.csv')

print("=== C8_IT_specific_genes.csv ===")
print(f"Total rows: {len(it_spec)}")
print(f"Columns: {list(it_spec.columns)}")
print(f"\nFirst 5 rows:")
print(it_spec.head().to_string())

# Count by tissue
print(f"\n--- By tissue ---")
print(it_spec['tissue'].value_counts())

# Count by lineage
print(f"\n--- By lineage ---")
print(it_spec['lineage'].value_counts())

# Unique genes
print(f"\n--- Unique genes: {it_spec['gene'].nunique()} ---")

# Check: is this from C5 (148 genes) or C3 (196 genes)?
print(f"\n--- Gene list sample ---")
print(sorted(it_spec['gene'].unique())[:20])

# Check if SOCS1, MX1, ISG15 are present (these are C3-only genes)
c3_only_check = ['SOCS1', 'SOCS3', 'MX1', 'ISG15', 'STAT2', 'IRF3', 'AICDA']
for g in c3_only_check:
    found = g in it_spec['gene'].values
    print(f"  {g}: {'FOUND' if found else 'NOT FOUND'} (C3-only gene)")

=== C8_IT_specific_genes.csv ===
Total rows: 122
Columns: ['tissue', 'lineage', 'gene', 'pathway', 'IT_pct', 'IT_p', 'IA_p']

First 5 rows:
  tissue  lineage   gene                          pathway  IT_pct      IT_p      IA_p
0  Liver  Myeloid    ATM                cancer_associated    43.8  0.025974  0.536797
1  Liver    CD8_T   BCL6  memory_t, tf_programs, stemness   163.1  0.004329  0.177489
2  Liver  Myeloid  CASP3                        apoptosis   112.4  0.025974  0.792208
3  Liver  Myeloid  CASP8                        apoptosis    77.9  0.041126  0.930736
4  Liver  Myeloid   CCR7                          naive_t   460.6  0.012907  0.779243

--- By tissue ---
tissue
Blood    85
Liver    37
Name: count, dtype: int64

--- By lineage ---
lineage
Myeloid    25
CD8_T      22
NK         22
B          18
CD4_T      18
PlasmaB    16
gdT         1
Name: count, dtype: int64

--- Unique genes: 75 ---

--- Gene list sample ---
['AIM2', 'ATM', 'ATP5F1A', 'ATP5F1B', 'BAK1', 'BATF', 'BCL6', 'B

In [4]:
# ========================================
# Also check C5 all_significant for IT-specific pattern
# ========================================
c5_all = pd.read_csv('/content/drive/MyDrive/ITLAS/results/version18-analysis/C5_genes/C5_all_significant_genes.csv')

print("=== C5_all_significant_genes.csv ===")
print(f"Shape: {c5_all.shape}")
print(f"Columns: {list(c5_all.columns)}")
print(c5_all.head(3).to_string())

# Check for IT-specific classification
for col in c5_all.columns:
    if 'pattern' in col.lower() or 'it_spec' in col.lower() or 'class' in col.lower():
        print(f"\n{col}: {c5_all[col].value_counts().to_dict()}")

# If comparison column exists, count NL→IT significant
if 'comparison' in c5_all.columns:
    nlit = c5_all[c5_all['comparison'] == 'NL→IT']
    print(f"\nNL→IT rows: {len(nlit)}")
    if 'tissue' in c5_all.columns:
        print(nlit['tissue'].value_counts())

=== C5_all_significant_genes.csv ===
Shape: (975, 18)
Columns: ['tissue', 'gene', 'lineage', 'pathway', 'comparison', 'stage1', 'stage2', 'mean_s1', 'mean_s2', 'pct_change', 'direction', 'p_value', 'sig', 'consistency', 'consist_pct', 'n_s1', 'n_s2', 'sig_tissue']
  tissue  gene  lineage                            pathway comparison stage1 stage2   mean_s1   mean_s2  pct_change direction   p_value sig consistency  consist_pct  n_s1  n_s2 sig_tissue
0  Liver   ATM  Myeloid                  cancer_associated      NL→IT     NL     IT  0.222100  0.319366        43.8         ↑  0.025974   *       32/36         88.9     6     6      Liver
1  Liver  BAK1  Myeloid                          apoptosis      NL→IT     NL     IT  0.050310  0.241250       379.5         ↑  0.002165  **       36/36        100.0     6     6      Liver
2  Liver  BATF  Myeloid  exhaustion, memory_t, tf_programs      NL→IT     NL     IT  0.164109  0.061665       -62.4         ↓  0.025974   *       32/36         88.9     6 

In [5]:
# ========================================
# IT-specific 정확한 숫자 재계산
# ========================================

# Method: C5 all significant에서 NL→IT sig인 것 중 NL→IA가 NS인 것
c5_all = pd.read_csv('/content/drive/MyDrive/ITLAS/results/version18-analysis/C5_genes/C5_all_significant_genes.csv')

# Step 1: NL→IT significant (p<0.05)
nlit = c5_all[c5_all['comparison'] == 'NL→IT'].copy()
print(f"NL→IT significant total: {len(nlit)}")
print(f"  Liver: {(nlit['tissue']=='Liver').sum()}")
print(f"  Blood: {(nlit['tissue']=='Blood').sum()}")

# Step 2: Get NL→IA results for same gene-lineage-tissue combos
nlia = c5_all[c5_all['comparison'] == 'NL→IA'].copy()
print(f"\nNL→IA significant total: {len(nlia)}")

# Step 3: For each NL→IT sig, check if NL→IA is also sig
# Need FULL C5 results (not just significant) to find NL→IA NS cases
# Read full C5 data
liver_c5 = pd.read_csv('/content/drive/MyDrive/ITLAS/results/version18-analysis/C5_genes/C5_genes_liver.csv')
blood_c5 = pd.read_csv('/content/drive/MyDrive/ITLAS/results/version18-analysis/C5_genes/C5_genes_blood.csv')

print(f"\nFull C5 liver: {liver_c5.shape}")
print(f"Full C5 blood: {blood_c5.shape}")
print(f"Columns: {list(liver_c5.columns)[:12]}")

# Check comparisons available
if 'comparison' in liver_c5.columns:
    print(f"\nLiver comparisons: {liver_c5['comparison'].unique()}")
    print(f"Blood comparisons: {blood_c5['comparison'].unique()}")

NL→IT significant total: 256
  Liver: 84
  Blood: 172

NL→IA significant total: 184

Full C5 liver: (7252, 17)
Full C5 blood: (7252, 17)
Columns: ['tissue', 'gene', 'lineage', 'pathway', 'comparison', 'stage1', 'stage2', 'mean_s1', 'mean_s2', 'pct_change', 'direction', 'p_value']

Liver comparisons: ['NL→IT' 'NL→IA' 'NL→AR' 'NL→CR' 'IT→IA' 'IA→AR' 'CR→AR']
Blood comparisons: ['NL→IT' 'NL→IA' 'NL→AR' 'NL→CR' 'IT→IA' 'IA→AR' 'CR→AR']


In [6]:
# ========================================
# Compute IT-specific from full C5 data
# ========================================

def compute_it_specific(full_df, tissue_name):
    """IT-specific = NL→IT sig (p<0.05) AND NL→IA NOT sig (p>=0.05)"""

    # NL→IT significant
    nlit = full_df[(full_df['comparison'] == 'NL→IT') & (full_df['p_value'] < 0.05)].copy()
    nlit_keys = set(zip(nlit['gene'], nlit['lineage']))

    # NL→IA results (ALL, not just significant)
    nlia = full_df[full_df['comparison'] == 'NL→IA'].copy()
    nlia_dict = {(r['gene'], r['lineage']): r['p_value']
                 for _, r in nlia.iterrows()}

    it_specific = []
    for _, row in nlit.iterrows():
        key = (row['gene'], row['lineage'])
        ia_p = nlia_dict.get(key, None)
        if ia_p is not None and ia_p >= 0.05:  # NL→IA NOT significant
            it_specific.append({
                'tissue': tissue_name,
                'gene': row['gene'],
                'lineage': row['lineage'],
                'IT_pct': row['pct_change'],
                'IT_p': row['p_value'],
                'IA_p': ia_p,
            })
        elif ia_p is None:
            # NL→IA comparison not available (e.g., missing data)
            it_specific.append({
                'tissue': tissue_name,
                'gene': row['gene'],
                'lineage': row['lineage'],
                'IT_pct': row['pct_change'],
                'IT_p': row['p_value'],
                'IA_p': None,
            })

    return pd.DataFrame(it_specific)

liver_it_spec = compute_it_specific(liver_c5, 'Liver')
blood_it_spec = compute_it_specific(blood_c5, 'Blood')

print(f"IT-specific from C5 (148 genes):")
print(f"  Liver: {len(liver_it_spec)}")
print(f"  Blood: {len(blood_it_spec)}")
print(f"  Total: {len(liver_it_spec) + len(blood_it_spec)}")

# Compare with C8
print(f"\nC8 file counts:")
print(f"  Liver: 37")
print(f"  Blood: 85")
print(f"  Total: 122")

# Compare with Results text
print(f"\nResults 2.1 claims:")
print(f"  Liver: 85")
print(f"  Blood: 169")
print(f"  Total: 254")

# Show by lineage
print(f"\n--- Liver IT-specific by lineage ---")
if len(liver_it_spec) > 0:
    print(liver_it_spec['lineage'].value_counts())

print(f"\n--- Blood IT-specific by lineage ---")
if len(blood_it_spec) > 0:
    print(blood_it_spec['lineage'].value_counts())

IT-specific from C5 (148 genes):
  Liver: 54
  Blood: 116
  Total: 170

C8 file counts:
  Liver: 37
  Blood: 85
  Total: 122

Results 2.1 claims:
  Liver: 85
  Blood: 169
  Total: 254

--- Liver IT-specific by lineage ---
lineage
Myeloid    13
PlasmaB    12
CD8_T      10
CD4_T       9
B           4
NK          4
gdT         2
Name: count, dtype: int64

--- Blood IT-specific by lineage ---
lineage
B          23
NK         22
Myeloid    22
CD8_T      20
CD4_T      17
PlasmaB    12
Name: count, dtype: int64


In [7]:
# ========================================
# Final count: gdT excluded
# ========================================
liver_final = liver_it_spec[liver_it_spec['lineage'] != 'gdT']
blood_final = blood_it_spec[blood_it_spec['lineage'] != 'gdT']

print(f"IT-specific (C5 148 genes, 6 lineages, gdT excluded):")
print(f"  Liver: {len(liver_final)}")
print(f"  Blood: {len(blood_final)}")
print(f"  Total: {len(liver_final) + len(blood_final)}")

# Also check C8 without gdT
it_c8 = pd.read_csv('/content/drive/MyDrive/ITLAS/results/version18-analysis/C8_characteristics/C8_IT_specific_genes.csv')
c8_no_gdt = it_c8[it_c8['lineage'] != 'gdT']
print(f"\nC8 (gdT excluded):")
print(f"  Liver: {(c8_no_gdt['tissue']=='Liver').sum()}")
print(f"  Blood: {(c8_no_gdt['tissue']=='Blood').sum()}")
print(f"  Total: {len(c8_no_gdt)}")

# Why C8 < C5 recomputed? Check if C8 has extra filtering
print(f"\n--- C5-recomputed Liver genes NOT in C8 ---")
c8_liver_keys = set(zip(
    c8_no_gdt[c8_no_gdt['tissue']=='Liver']['gene'],
    c8_no_gdt[c8_no_gdt['tissue']=='Liver']['lineage']
))
c5_liver_keys = set(zip(liver_final['gene'], liver_final['lineage']))
diff = c5_liver_keys - c8_liver_keys
print(f"  Count: {len(diff)}")
for g, l in sorted(list(diff))[:10]:
    row = liver_final[(liver_final['gene']==g) & (liver_final['lineage']==l)]
    print(f"  {g}/{l}: IT_p={row['IT_p'].values[0]:.4f}, IA_p={row['IA_p'].values[0]:.4f}")

IT-specific (C5 148 genes, 6 lineages, gdT excluded):
  Liver: 52
  Blood: 116
  Total: 168

C8 (gdT excluded):
  Liver: 36
  Blood: 85
  Total: 121

--- C5-recomputed Liver genes NOT in C8 ---
  Count: 16
  BATF/Myeloid: IT_p=0.0260, IA_p=0.0821
  BIRC5/Myeloid: IT_p=0.0403, IA_p=0.0732
  CD69/B: IT_p=0.0173, IA_p=0.0519
  FCER1G/PlasmaB: IT_p=0.0173, IA_p=0.0519
  GZMB/PlasmaB: IT_p=0.0087, IA_p=0.0519
  IL2RA/B: IT_p=0.0080, IA_p=0.0821
  IRF4/Myeloid: IT_p=0.0152, IA_p=0.0519
  JAK1/CD4_T: IT_p=0.0411, IA_p=0.0823
  MTOR/CD4_T: IT_p=0.0087, IA_p=0.0519
  SDHB/CD4_T: IT_p=0.0411, IA_p=0.0519


In [9]:
# ========================================
# Generate S2 Sheet 2 data: IT-specific combinations
# ========================================
import pandas as pd

# Combine liver + blood, exclude gdT
s2_sheet2 = pd.concat([liver_final, blood_final], ignore_index=True)

# Add pathway info from C5 full data
liver_c5 = pd.read_csv('/content/drive/MyDrive/ITLAS/results/version18-analysis/C5_genes/C5_genes_liver.csv')
blood_c5 = pd.read_csv('/content/drive/MyDrive/ITLAS/results/version18-analysis/C5_genes/C5_genes_blood.csv')

# Get pathway info
c5_full = pd.concat([liver_c5, blood_c5])
c5_nlit = c5_full[c5_full['comparison'] == 'NL→IT'].copy()

# Merge to get additional columns (direction, consistency)
merged = s2_sheet2.merge(
    c5_nlit[['tissue', 'gene', 'lineage', 'direction', 'consistency', 'pathway']].rename(
        columns={'pathway': 'pathway_c5'}),
    on=['tissue', 'gene', 'lineage'],
    how='left'
)

# Use pathway from original if pathway_c5 exists
if 'pathway' in merged.columns:
    merged['pathway_final'] = merged['pathway'].fillna(merged.get('pathway_c5', ''))
else:
    merged['pathway_final'] = merged.get('pathway_c5', '')

# Clean up and sort
output = merged[['tissue', 'lineage', 'gene', 'pathway_final', 'direction',
                  'IT_pct', 'IT_p', 'IA_p', 'consistency']].copy()
output.columns = ['Tissue', 'Lineage', 'Gene', 'Pathway', 'Direction',
                   'IT_pct_change', 'NL_IT_p', 'NL_IA_p', 'Consistency']
output = output.sort_values(['Tissue', 'Lineage', 'Gene']).reset_index(drop=True)

print(f"S2 Sheet 2 total rows: {len(output)}")
print(f"  Liver: {(output['Tissue']=='Liver').sum()}")
print(f"  Blood: {(output['Tissue']=='Blood').sum()}")
print(f"\nSample rows:")
print(output.head(10).to_string())

# Save to CSV for later xlsx conversion
output_path = '/content/drive/MyDrive/ITLAS/results/version18-analysis/S2_IT_specific_combinations.csv'
output.to_csv(output_path, index=False)
print(f"\n✅ Saved: {output_path}")

S2 Sheet 2 total rows: 168
  Liver: 52
  Blood: 116

Sample rows:
  Tissue Lineage      Gene                        Pathway Direction  IT_pct_change   NL_IT_p   NL_IA_p Consistency
0  Blood       B      AIM2                   inflammasome         ↑           69.8  0.017677  0.412698       32/35
1  Blood       B   ATP5F1A                         oxphos         ↑           51.6  0.030303  0.285714       31/35
2  Blood       B      BCL2                      apoptosis         ↑           81.5  0.005051  0.063492       34/35
3  Blood       B     CD274                 immune_evasion         ↑          478.7  0.043308  0.466854       32/35
4  Blood       B      CDK4  cell_cycle, cancer_associated         ↑          120.3  0.005051  0.190476       34/35
5  Blood       B    DNMT3A                    epigenetics         ↑          227.6  0.022752  0.065064       32/35
6  Blood       B  EIF4EBP1                      il15_mtor         ↑           56.8  0.017677  0.904762       32/35
7  Blood      

In [10]:
# ========================================
# Verify: check key claims in Results against this data
# ========================================
print("="*60)
print("VERIFICATION: Key Results claims vs S2 Sheet 2")
print("="*60)

# Check specific genes mentioned in Results 2.3-2.4
key_checks = [
    ('Liver', 'CD4_T', 'TOX', 'Liver-specific exhaustion'),
    ('Liver', 'CD4_T', 'TOX2', 'Liver-specific exhaustion'),
    ('Liver', 'CD4_T', 'LAYN', 'Liver-specific exhaustion'),
    ('Liver', 'CD8_T', 'TOX', 'Liver-specific exhaustion'),
    ('Liver', 'CD8_T', 'BCL6', 'Liver-specific stemness'),
    ('Liver', 'CD4_T', 'CTLA4', 'Liver-specific Treg'),
]

print(f"\n{'Tissue':<7} {'Lineage':<9} {'Gene':<8} {'In S2?':>7} {'IT_p':>8} {'IA_p':>8} | Note")
print("-"*65)
for tissue, lin, gene, note in key_checks:
    row = output[(output['Tissue']==tissue) & (output['Lineage']==lin) & (output['Gene']==gene)]
    if len(row) > 0:
        r = row.iloc[0]
        print(f"{tissue:<7} {lin:<9} {gene:<8} {'YES':>7} {r['NL_IT_p']:>8.4f} {r['NL_IA_p']:>8.4f} | {note}")
    else:
        # Check if gene is in C5 148-gene list at all
        in_c5 = gene in c5_nlit['gene'].values
        print(f"{tissue:<7} {lin:<9} {gene:<8} {'NO':>7} {'—':>8} {'—':>8} | {note} ⚠️ {'(not in C5 148)' if not in_c5 else '(in C5 but not IT-specific)'}")

VERIFICATION: Key Results claims vs S2 Sheet 2

Tissue  Lineage   Gene      In S2?     IT_p     IA_p | Note
-----------------------------------------------------------------
Liver   CD4_T     TOX          YES   0.0087   0.1255 | Liver-specific exhaustion
Liver   CD4_T     TOX2          NO        —        — | Liver-specific exhaustion ⚠️ (in C5 but not IT-specific)
Liver   CD4_T     LAYN         YES   0.0087   0.1255 | Liver-specific exhaustion
Liver   CD8_T     TOX          YES   0.0022   0.1255 | Liver-specific exhaustion
Liver   CD8_T     BCL6         YES   0.0043   0.1775 | Liver-specific stemness
Liver   CD4_T     CTLA4         NO        —        — | Liver-specific Treg ⚠️ (in C5 but not IT-specific)


In [11]:
# 에러발견: TOX2와 CTLA4 — IT-specific이 아님
# (둘 다 C5 148-gene에 포함되어 있지만, IT-specific 기준(NL→IT sig + NL→IA NS)을 충족하지 못함)
# 원인을 확인해야...

In [12]:
# ========================================
# TOX2 and CTLA4: 왜 IT-specific이 아닌가?
# ========================================
check_genes = ['TOX2', 'CTLA4', 'TOX', 'LAYN', 'TIGIT', 'RORC']

print("="*80)
print("WHY NOT IT-SPECIFIC? Checking NL→IT and NL→IA p-values")
print("="*80)

for gene in check_genes:
    print(f"\n--- {gene} ---")
    for tissue, c5_df in [('Liver', liver_c5), ('Blood', blood_c5)]:
        for comp in ['NL→IT', 'NL→IA']:
            rows = c5_df[(c5_df['gene'] == gene) & (c5_df['comparison'] == comp)]
            if len(rows) > 0:
                for _, r in rows.iterrows():
                    sig = '★' if r['p_value'] < 0.05 else ('†' if r['p_value'] < 0.10 else 'NS')
                    print(f"  {tissue:<6} {comp:<7} {r['lineage']:<9} pct={r['pct_change']:>+7.1f}% "
                          f"p={r['p_value']:.4f} {sig}  consist={r.get('consistency','?')}")

WHY NOT IT-SPECIFIC? Checking NL→IT and NL→IA p-values

--- TOX2 ---
  Liver  NL→IT   B         pct=  +52.6% p=0.6404 NS  consist=20/30
  Liver  NL→IT   CD8_T     pct=  +26.4% p=0.6991 NS  consist=21/36
  Liver  NL→IT   Myeloid   pct=+99999.0% p=0.0740 †  consist=36/36
  Liver  NL→IT   NK        pct=  +15.6% p=0.8182 NS  consist=16/36
  Liver  NL→IT   CD4_T     pct= +242.1% p=0.0087 ★  consist=34/36
  Liver  NL→IT   PlasmaB   pct= +222.1% p=0.1198 NS  consist=24/30
  Liver  NL→IT   gdT       pct=  +71.3% p=0.2571 NS  consist=18/24
  Liver  NL→IA   B         pct=  +69.1% p=0.3142 NS  consist=21/30
  Liver  NL→IA   CD8_T     pct= +199.2% p=0.1255 NS  consist=24/30
  Liver  NL→IA   Myeloid   pct=+99999.0% p=0.1364 NS  consist=30/30
  Liver  NL→IA   NK        pct=  +18.8% p=0.6623 NS  consist=18/30
  Liver  NL→IA   CD4_T     pct= +250.8% p=0.0303 ★  consist=27/30
  Liver  NL→IA   PlasmaB   pct= +170.3% p=0.1670 NS  consist=24/30
  Liver  NL→IA   gdT       pct=  +18.4% p=1.0000 NS  consist=

In [13]:
# ========================================
# Results 2.3 claim verification — ALL liver-specific genes
# ========================================
print("="*80)
print("RESULTS 2.3 LIVER-SPECIFIC CLAIMS vs DATA")
print("="*80)

claims_23 = [
    # (gene, lineage, claim_type, expected_liver_pct)
    ('TOX',   'CD4_T',   'exhaustion',  100.7),
    ('TOX2',  'CD4_T',   'exhaustion',  242.1),
    ('LAYN',  'CD4_T',   'exhaustion',  430.5),
    ('CTLA4', 'CD4_T',   'Treg',        171.1),
    ('TIGIT', 'CD4_T',   'Treg',        123.8),
    ('TOX',   'CD8_T',   'exhaustion',   86.6),
    ('BCL6',  'CD8_T',   'stemness',    163.1),
    ('TIGIT', 'CD8_T',   'exhaustion',  151.5),
    ('RORC',  'CD4_T',   'Treg_down',   -49.4),
    ('TOX',   'Myeloid', 'aberrant',    1856.8),
    ('CCR7',  'Myeloid', 'migration',    460.6),
]

print(f"\n{'Gene':<7} {'Lineage':<9} {'Claim':<12} {'Exp_pct':>8} | "
      f"{'L_IT_pct':>9} {'L_IT_p':>8} {'L_IA_p':>8} | "
      f"{'B_IT_pct':>9} {'B_IT_p':>8} | {'IT-spec?'}")
print("-"*105)

for gene, lin, claim, exp_pct in claims_23:
    # Liver NL→IT
    lr_it = liver_c5[(liver_c5['gene']==gene) & (liver_c5['lineage']==lin) & (liver_c5['comparison']=='NL→IT')]
    lr_ia = liver_c5[(liver_c5['gene']==gene) & (liver_c5['lineage']==lin) & (liver_c5['comparison']=='NL→IA')]

    l_pct = lr_it['pct_change'].values[0] if len(lr_it) > 0 else None
    l_p_it = lr_it['p_value'].values[0] if len(lr_it) > 0 else None
    l_p_ia = lr_ia['p_value'].values[0] if len(lr_ia) > 0 else None

    # Blood NL→IT
    br_it = blood_c5[(blood_c5['gene']==gene) & (blood_c5['lineage']==lin) & (blood_c5['comparison']=='NL→IT')]
    b_pct = br_it['pct_change'].values[0] if len(br_it) > 0 else None
    b_p_it = br_it['p_value'].values[0] if len(br_it) > 0 else None

    # IT-specific?
    is_it_spec = (l_p_it is not None and l_p_it < 0.05 and
                  l_p_ia is not None and l_p_ia >= 0.05)
    spec_label = "✅ YES" if is_it_spec else "❌ NO"

    # If not IT-specific, what pattern?
    if not is_it_spec and l_p_it is not None and l_p_it < 0.05:
        if l_p_ia is not None and l_p_ia < 0.05:
            spec_label += " (chronic)"
        else:
            spec_label += " (???)"

    l_pct_str = f"{l_pct:>+8.1f}%" if l_pct is not None else "    N/A"
    l_p_it_str = f"{l_p_it:>8.4f}" if l_p_it is not None else "     N/A"
    l_p_ia_str = f"{l_p_ia:>8.4f}" if l_p_ia is not None else "     N/A"
    b_pct_str = f"{b_pct:>+8.1f}%" if b_pct is not None else "    N/A"
    b_p_str = f"{b_p_it:>8.4f}" if b_p_it is not None else "     N/A"

    print(f"{gene:<7} {lin:<9} {claim:<12} {exp_pct:>+7.1f}% | "
          f"{l_pct_str} {l_p_it_str} {l_p_ia_str} | "
          f"{b_pct_str} {b_p_str} | {spec_label}")

RESULTS 2.3 LIVER-SPECIFIC CLAIMS vs DATA

Gene    Lineage   Claim         Exp_pct |  L_IT_pct   L_IT_p   L_IA_p |  B_IT_pct   B_IT_p | IT-spec?
---------------------------------------------------------------------------------------------------------
TOX     CD4_T     exhaustion    +100.7% |   +100.7%   0.0087   0.1255 |    -44.3%   0.7551 | ✅ YES
TOX2    CD4_T     exhaustion    +242.1% |   +242.1%   0.0087   0.0303 |    -38.2%   0.4318 | ❌ NO (chronic)
LAYN    CD4_T     exhaustion    +430.5% |   +430.5%   0.0087   0.1255 |    -46.6%   0.3290 | ✅ YES
CTLA4   CD4_T     Treg          +171.1% |   +171.1%   0.0043   0.0303 |    -53.6%   0.0732 | ❌ NO (chronic)
TIGIT   CD4_T     Treg          +123.8% |   +123.8%   0.0152   0.0087 |    -23.2%   0.7551 | ❌ NO (chronic)
TOX     CD8_T     exhaustion     +86.6% |    +86.6%   0.0022   0.1255 |    -36.7%   0.3434 | ✅ YES
BCL6    CD8_T     stemness      +163.1% |   +163.1%   0.0043   0.1775 |    -48.4%   0.3998 | ✅ YES
TIGIT   CD8_T     exhaustion 

In [ ]:
========================================
# S2 Sheet 2: Populate with actual IT-specific data
# ========================================

In [14]:
# ========================================
# S2 Sheet 2: Populate with actual IT-specific data
# ========================================
from openpyxl import load_workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
import pandas as pd

# Read the IT-specific CSV
csv_path = '/content/drive/MyDrive/ITLAS/results/version18-analysis/S2_IT_specific_combinations.csv'
data = pd.read_csv(csv_path)
print(f"Loaded: {len(data)} rows")
print(f"  Liver: {(data['Tissue']=='Liver').sum()}")
print(f"  Blood: {(data['Tissue']=='Blood').sum()}")

# Load existing S2 (upload the original to Colab or use Drive path)
# If S2 is in Drive:
import shutil
s2_src = '/content/drive/MyDrive/ITLAS/results/version18-analysis/Supplementary_Table_S2.xlsx'

# Try multiple possible locations
import os
for candidate in [
    s2_src,
    '/content/drive/MyDrive/ITLAS/Supplementary_Table_S2.xlsx',
]:
    if os.path.exists(candidate):
        s2_src = candidate
        break

if not os.path.exists(s2_src):
    print("⚠️ S2 xlsx not found in Drive. Upload it to Colab and set s2_src path.")
else:
    wb = load_workbook(s2_src)
    print(f"Loaded S2: sheets={wb.sheetnames}")

    # Remove old Sheet 2 if exists
    if 'S2_IT_Specific_Combinations' in wb.sheetnames:
        del wb['S2_IT_Specific_Combinations']

    ws2 = wb.create_sheet("S2_IT_Specific_Combinations", index=1)

    # Styles
    header_font = Font(name='Arial', bold=True, size=10)
    header_fill = PatternFill('solid', fgColor='D5E8F0')
    title_font = Font(name='Arial', bold=True, size=11)
    subtitle_font = Font(name='Arial', italic=True, size=9)
    data_font = Font(name='Arial', size=9)
    liver_fill = PatternFill('solid', fgColor='FFF2E5')  # light orange
    blood_fill = PatternFill('solid', fgColor='E5F0FF')  # light blue
    border = Border(
        bottom=Side(style='thin', color='CCCCCC'),
        top=Side(style='thin', color='CCCCCC'),
        left=Side(style='thin', color='CCCCCC'),
        right=Side(style='thin', color='CCCCCC')
    )

    # Title
    ws2['A1'] = 'Supplementary Table S2b. IT-Specific Gene-Lineage Combinations (C5 148-Gene Panel)'
    ws2['A1'].font = title_font
    ws2.merge_cells('A1:I1')

    ws2['A2'] = ('IT-specific: NL→IT significant (p<0.05) AND NL→IA non-significant (p≥0.05). '
                 f'{(data["Tissue"]=="Liver").sum()} liver + {(data["Tissue"]=="Blood").sum()} blood = {len(data)} total. '
                 '148 genes × 6 lineages (gdT excluded). Donor-level Mann-Whitney U tests.')
    ws2['A2'].font = subtitle_font
    ws2.merge_cells('A2:I2')

    # Headers (row 4)
    headers = ['Tissue', 'Lineage', 'Gene', 'Pathway', 'Direction',
               'IT % Change', 'NL→IT p', 'NL→IA p', 'Consistency']
    for col, h in enumerate(headers, 1):
        cell = ws2.cell(row=4, column=col, value=h)
        cell.font = header_font
        cell.fill = header_fill
        cell.alignment = Alignment(horizontal='center', wrap_text=True)
        cell.border = border

    # Data rows (starting row 5)
    # Sort: Liver first, then Blood; within each, by Lineage then Gene
    data_sorted = data.sort_values(['Tissue', 'Lineage', 'Gene'],
                                    key=lambda x: x.map({'Liver': 0, 'Blood': 1}) if x.name == 'Tissue' else x)

    for idx, (_, row) in enumerate(data_sorted.iterrows()):
        r = idx + 5  # Excel row
        fill = liver_fill if row['Tissue'] == 'Liver' else blood_fill

        values = [
            row['Tissue'], row['Lineage'], row['Gene'], row['Pathway'],
            row['Direction'],
            round(row['IT_pct_change'], 1),
            round(row['NL_IT_p'], 4),
            round(row['NL_IA_p'], 4) if pd.notna(row['NL_IA_p']) else 'N/A',
            row['Consistency'] if pd.notna(row.get('Consistency', None)) else ''
        ]

        for col, val in enumerate(values, 1):
            cell = ws2.cell(row=r, column=col, value=val)
            cell.font = data_font
            cell.fill = fill
            cell.border = border
            if col in [6, 7, 8]:  # numeric columns
                cell.alignment = Alignment(horizontal='right')
                if col == 6:
                    cell.number_format = '+0.0;-0.0;0.0'
                elif col in [7, 8] and isinstance(val, float):
                    cell.number_format = '0.0000'

    # Column widths
    for col_letter, width in [('A',8),('B',10),('C',10),('D',35),('E',9),('F',12),('G',10),('H',10),('I',12)]:
        ws2.column_dimensions[col_letter].width = width

    # Freeze panes
    ws2.freeze_panes = 'A5'

    # Save
    out_path = '/content/drive/MyDrive/ITLAS/results/version18-analysis/Supplementary_Table_S2_v2.xlsx'
    wb.save(out_path)
    print(f"\n✅ Saved: {out_path}")
    print(f"   Sheets: {wb.sheetnames}")
    print(f"   Sheet 2 rows: {len(data)} data + 4 header = {len(data)+4}")

Loaded: 168 rows
  Liver: 52
  Blood: 116
Loaded S2: sheets=['S2_Gene_Candidates', 'Legend']

✅ Saved: /content/drive/MyDrive/ITLAS/results/version18-analysis/Supplementary_Table_S2_v2.xlsx
   Sheets: ['S2_Gene_Candidates', 'S2_IT_Specific_Combinations', 'Legend']
   Sheet 2 rows: 168 data + 4 header = 172
